In [ ]:
#%pip install torch torchvision

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
import pandas as pd
import joblib
from torch.utils.data import TensorDataset, DataLoader

# ============================================================
# Dry Bean Dataset (UCI, ID 602)
# 13.611 Proben | 16 numerische Features | 7 Klassen
# Klassen: Seker, Barbunya, Bombay, Cali, Dermason, Horoz, Sira
# ============================================================

df= pd.read_csv('Dry_Bean_Dataset.csv')
print(f"Datensatz: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
print(df.head())
# Klassen in der Spalte 'Class' eindeutig identifizieren
print(df['Class'].unique()) 

In [ ]:
# Features (X) und Target (y) vorbereiten
X = df.drop('Class', axis=1).values      
y = df['Class'].values   

# LabelEncoder: Text-Klassen -> Integer
le = LabelEncoder()
y = le.fit_transform(y)

# -- NEU gegenüber Iris: StandardScaler --
# Die 16 Features haben sehr unterschiedliche Skalen
# (z.B. Area im Tausenderbereich, ShapeFactor < 1)
# Ohne Skalierung konvergiert das Netz schlecht oder gar nicht.
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
# Aufteilen: 80% Train, davon 20% Validierung, 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=0, stratify=y_train
)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

# Umwandeln in PyTorch-Tensoren
X_train = torch.from_numpy(X_train).float()
X_test  = torch.from_numpy(X_test).float()
y_train = torch.from_numpy(y_train).long()
y_test  = torch.from_numpy(y_test).long()
X_val   = torch.from_numpy(X_val).float()
y_val   = torch.from_numpy(y_val).long()

# DataLoader erstellen
train_dataset = TensorDataset(X_train, y_train)
batch_size = 64  # größer als bei Iris, weil Datensatz ~90x größer
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# ============================================================
# Neuronales Netz
# Experimentier-Stellschrauben:
#   - Anzahl Neuronen in den Hidden Layers
#   - Aktivierungsfunktion (ReLU, Tanh, LeakyReLU, GELU ...)
#   - Dropout-Rate
#   - Optimizer (SGD, Adam, AdamW, RMSprop ...)
#   - Batch-Size (oben)
#   - Lernrate
# ============================================================

net = nn.Sequential(
    nn.Linear(16, 64),     # 16 Features -> Hidden Layer 1
    nn.ReLU(),
    nn.Dropout(0.2),       # <-- hier Dropout-Rate variieren (0.0 bis 0.5)
    nn.Linear(64, 32),     # Hidden Layer 2
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(32, 7)       # 7 Klassen
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(net.parameters(), lr=0.001)

# Training
num_epochs = 50
net.train()
for epoch in range(num_epochs):
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = net(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

    # Validierung nach jeder Epoche
    net.eval()
    with torch.no_grad():
        val_out  = net(X_val)
        val_loss = criterion(val_out, y_val)
        _, val_pred = torch.max(val_out, 1)
        val_acc = accuracy_score(y_val, val_pred)
    net.train()

    if epoch % 5 == 0 or epoch == num_epochs - 1:
        print(f'Epoch {epoch:3d} | Train-Loss: {loss.item():.4f} | Val-Loss: {val_loss.item():.4f} | Val-Acc: {val_acc:.4f}')

In [ ]:
# Evaluation auf Testdaten
net.eval()
with torch.no_grad():
    outputs = net(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = accuracy_score(y_test, predicted)
    print(f'\nTestgenauigkeit: {accuracy:.4f}')

# Modell speichern
torch.save(net.state_dict(), 'drybean_net.pth')
print("Trainiertes Netz abgespeichert als 'drybean_net.pth'")
joblib.dump(le, 'label_encoder.pkl')
joblib.dump(scaler, 'scaler.pkl')  # Scaler muss mit gespeichert werden!

In [ ]:
# Wiederladen des trainierten Netzes
import torch
import torch.nn as nn
import joblib

le     = joblib.load('label_encoder.pkl')
scaler = joblib.load('scaler.pkl')

# Architektur muss identisch sein!
net = nn.Sequential(
    nn.Linear(16, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(32, 7)
)
net.load_state_dict(torch.load('drybean_net.pth', map_location=torch.device('cpu')))

for k, v in net.named_parameters():
    print(k, v.shape)

net.eval()

In [ ]:
# Vorhersage mit dem trainierten Netz
# Interaktive Eingabe der 16 Merkmale

import numpy as np

feature_names = [
    'Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength',
    'AspectRatio', 'Eccentricity', 'ConvexArea', 'EquivDiameter',
    'Extent', 'Solidity', 'Roundness', 'Compactness',
    'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4'
]

print("Geben Sie die 16 Merkmale ein:")
values = []
for name in feature_names:
    val = float(input(f"{name}: "))
    values.append(val)

# Wichtig: Skalierung mit dem gespeicherten Scaler anwenden!
inputs_scaled = scaler.transform(np.array([values]))
inputs = torch.tensor(inputs_scaled, dtype=torch.float32)

with torch.no_grad():
    outputs = net(inputs)
    _, predicted = torch.max(outputs, 1)

print(f"Klasse: {le.inverse_transform([predicted.item()])[0]}")

In [ ]:
# Klassenwahrscheinlichkeiten anzeigen

import torch.nn.functional as F

with torch.no_grad():
    output = net(inputs)
    _, max_index = torch.max(output, 1)
    print(f"Vorhergesagte Klasse: {max_index.item()} ({le.inverse_transform([max_index.item()])[0]})")

probabilities = F.softmax(output, dim=1)
print("\nKlassenwahrscheinlichkeiten:")
for i, (cls, prob) in enumerate(zip(le.classes_, probabilities.numpy()[0])):
    print(f"  {cls:10s}  {prob:.4f}")